# LAB2 — Comparação de Estratégias de Chunking

## Objetivos da Aula
1. **Aplicar 5 estratégias diferentes** de chunking no mesmo documento jurídico
2. **Visualizar as fronteiras de chunk** e identificar cortes ruins
3. **Medir chunk_size médio, mín, máx e overlap real** de cada estratégia
4. **Comparar quantitativamente** com gráficos e tabelas
5. **Selecionar a melhor estratégia** por tipo de caso de uso

---

## As 5 Estratégias de Chunking

| # | Estratégia | Descrição | Vantagens | Desvantagens |
|---|---|---|---|---|
| **1** | Fixed-Size | Chunks de tamanho fixo com overlap | Simples, previsível | Ignora limites semânticos |
| **2** | Recursive Character | Divide por separadores em ordem | Respeita estrutura | Pode gerar chunks grandes |
| **3** | Semantic | Usa embeddings para detectar breakpoints | Mais inteligente | Lento, requer embedding model |
| **4** | Sentence-Window | Cada sentença é unidade, contexto na janela | Granular, contextualizdo | Complexo, 2 índices |
| **5** | Document-Aware (Headers) | Explora hierarquia de headers | Respeita semantics real | Requer Markdown estruturado |

---

## Diagrama do Experimento Controlado

```
┌────────────────────────────────────────────────────────────────┐
│           TEXTO JURÍDICO ÚNICO (~2000 chars)                  │
│  [Acórdão: EMENTA + RELATÓRIO + FUNDAMENTAÇÃO + DISPOSITIVO]  │
└────────────────────────────────────────────────────────────────┘
                              │
                ┌─────────────┼─────────────┐
                │             │             │
         ┌──────▼──────┐ ┌───▼────────┐ ┌──▼──────────────┐
         │Strategy 1-2 │ │ Strategy 3 │ │ Strategy 4-5   │
         │ (Char-based)│ │ (Semantic) │ │ (Struct-aware) │
         └──────┬──────┘ └───┬────────┘ └──┬──────────────┘
                │            │             │
         ┌──────▼────────────▼─────────────▼────────┐
         │     ANÁLISE COMPARATIVA                  │
         │  • n_chunks                              │
         │  • chunk_size (média, min, max)          │
         │  • cortes_ruins (chunks\\que cortam em   │
         │     meio de palavra ou sentença)        │
         │  • tempo de execução                     │
         │  • overlap efetivo (chars repetidos)    │
         └──────┬────────────────────────────────────┘
                │
         ┌──────▼──────────────────┐
         │  VISUALIZAÇÃO & TABELAS │
         │  • DataFrame            │
         │  • Gráficos             │
         │  • Recomendações        │
         └───────────────────────────┘
```

In [2]:
# [CÉLULA 1] Instalação de dependências
%pip install nltk 
%pip install pandas matplotlib numpy langchain langchain-text-splitters langchain-experimental
%pip install sentence-transformers

# Importar bibliotecas
import os
import time
import warnings
warnings.filterwarnings('ignore')

# LangChain
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter
)

# Semantic chunking (experimental)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

# Sentence-Window Parser — implementação LangChain + NLTK (sem LlamaIndex)
from langchain_core.documents import Document as LCDocument

# Utilitários
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# NLTK para análise de sentenças
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize

print('✓ Todas as dependências instaladas')
print(f'  • LangChain: chunking com 5 estratégias (incl. sentence-window Python puro)')
print(f'  • NLTK: tokenização de sentenças (sent_tokenize)')
print(f'  • Embeddings BGE-M3 via Ollama (Aula 1): semantic chunking')

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 1.9 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.6 MB 1.6 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.6 MB 1.6 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 1.4 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 1.4 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 949.2 kB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ Todas as dependências instaladas
  • LangChain: chunking com 5 estratégias (incl. sentence-window Python puro)
  • NLTK: tokenizaçã

## Seção 1: Texto de Referência

Acórdão jurídico longo (~2000 caracteres) que será usado em TODAS as estratégias.
Mantém-se o **MESMO TEXTO** para comparação controlada.

In [11]:
# [CÉLULA 2] Definir texto de referência jurídico

TEXTO_ACORDAO = """
# ACÓRDÃO EM HABEAS CORPUS

## EMENTA

Habeas Corpus. Violação de direitos fundamentais. Condenação proferida sem respeito integral ao devido processo legal. Nulidade processual por falta de oportunidade de apresentação de contraditório e ampla defesa. Inadequação da pena aplicada pelo tribunal de origem. Provimento parcial do recurso. Direitos fundamentais ao contraditório, ampla defesa e ao processo justo. Lei nº 11.698/2008. Configuração de cerceamento de defesa por ausência de intimação pessoal da Defensoria Pública. Prejuízo manifesto demonstrado. Dosimetria da pena revista ante a ausência de fundamentação idônea na primeira fase. Concessão parcial da ordem.


## RELATÓRIO

Trata-se de *habeas corpus* substitutivo de recurso próprio, com pedido de liminar, impetrado em favor de **Fulano de Beltrando de Ciclano**, contra decisão proferida pelo Tribunal de Justiça do Estado que, ao julgar a apelação criminal da acusação, condenou o paciente pelo crime de roubo qualificado (art. 157, § 2º, inciso II, do Código Penal), sustentando vício processual grave na execução do julgamento realizado em primeira instância.

A defesa técnica, exercida pela Defensoria Pública, alega em suas razões iniciais não ter tido oportunidade adequada de apresentar contrarrazões e argumentações no prazo legal estabelecido pelos códigos processuais federais, haja vista a ausência de intimação pessoal do defensor público responsável pelo caso, violando a prerrogativa prevista no art. 128, inciso I, da Lei Complementar nº 80/1994.

O paciente foi condenado à pena de 8 anos de reclusão em regime inicial fechado, além do pagamento de 80 dias-multa, condenação que se busca reformar neste recurso extraordinário. Sustenta-se, em suma:

1. A nulidade absoluta dos atos processuais praticados após a fase do art. 402 do Código de Processo Penal;
2. O manifesto cerceamento de defesa pela falta de manifestação sobre documentos cruciais juntados pelo Ministério Público de forma tardia;
3. A ausência de fundamentação idônea na fixação da pena-base acima do mínimo legal.

O pedido de liminar foi indeferido pelo Relator originário às fls. 142-145. As informações foram devidamente prestadas pela autoridade coatora (fls. 150-168), acompanhadas dos documentos de praxe. A douta Procuradoria-Geral da República manifestou-se, em parecer escrito (fls. 172-180), pela denegação da ordem, sob o argumento de que a matéria demandaria revolvimento fático-probatório, incompatível com a via estreita do *habeas corpus*.

É o relatório necessário. Passo ao voto.

## FUNDAMENTAÇÃO

### I. Das Preliminares: Do Cabimento do Habeas Corpus e do Cerceamento de Defesa

O direito ao contraditório e à ampla defesa é garantido constitucionalmente no art. 5º, inciso LV da Constituição Federal de 1988. Precedentes jurisprudenciais consolidados desta Corte Suprema: HC 2021.001, HC 2020.456, HC 2019.789. A jurisprudência nacional e internacional consolidam que violações processuais nessa magnitude ensejam concessão de *habeas corpus* sem questionamento sobre o mérito condenatório. Neste sentido, pronunciou-se a Corte Europeia de Direitos Humanos no Caso *"Smith vs. Reino Unido"*, afirmando ser inalienável o direito ao processo justo e à igualdade de armas no cenário processual penal.

No caso em tela, verifica-se que a ausência de intimação pessoal do defensor público para a audiência de instrução e julgamento gerou prejuízo evidente e insanável à defesa do paciente. O princípio do *pas de de nullité sans grief*, embora aplicável no direito processual penal brasileiro, deve ser mitigado quando a ausência do ato essencial impede a própria manifestação técnica da defesa, gerando uma condenação baseada exclusivamente em provas unilaterais.

> "A garantia da ampla defesa não se resume à presença formal de um defensor no recinto, mas sim à oportunidade real, efetiva e tempestiva de influenciar a convicção do magistrado através de argumentos e contraprovas." (Min. Relator, HC 2022.891)

### II. Do Mérito: Da Tipicidade e das Circunstâncias Atenuantes

Quanto ao mérito condenatório, embora reconheçamos a tipicidade da conduta delituosa e a materialidade do crime de roubo — devidamente comprovadas pelo auto de exibição e apreensão e pelos depoimentos das testemunhas oculares colhidos na fase inquisitorial —, a pena aplicada mostra-se desproporcional ao caso concreto analisado, especialmente considerando as circunstâncias atenuantes comprovadas nos autos e a reincidência de natureza leve.

A Súmula nº 18/2019 desta Corte estabelece claramente que a individualização da pena deve observar critérios racionais e proporcionais. O tribunal de origem não observou tal orientação ao valorar negativamente a personalidade do agente com base em inquéritos policiais em andamento, o que confronta diretamente a Súmula nº 444 do Superior Tribunal de Justiça.

### III. Da Dosimetria da Pena e Ajuste de Regime

No tocante à dosimetria da pena, a sentença condenatória não observou critério racional e sistemático na individualização, deixando de considerar antecedentes criminais irrelevantes e circunstâncias sociais do acusado. Conforme pacífica jurisprudência consolidada, a Corte pode revisar decisões que violem direitos fundamentais, sendo competente para reformar sentenças flagrantemente injustas ou processualizadas de forma viciada.

Analisando as três fases da dosimetria (art. 68 do Código Penal):

* **Primeira Fase:** A pena-base foi fixada em 5 anos de reclusão (1 ano acima do mínimo legal). Afasta-se a valoração negativa das "consequências do crime", visto que o trauma psicológico da vítima, embora lamentável, é inerente ao tipo penal do roubo majorado pelo emprego de violência ou grave ameaça.
* **Segunda Fase:** Reconhece-se a atenuante da menoridade relativa (o paciente contava com 19 anos à época dos fatos), compensando-a integralmente com a agravante da reincidência genérica.
* **Terceira Fase:** Mantém-se a causa de aumento referente ao concurso de agentes no patamar mínimo de 1/3, à míngua de fundamentação concreta para elevação superior.

Dessa forma, o redimensionamento da reprimenda é medida de rigor, impondo-se também a readequação do regime inicial de cumprimento de pena para o semiaberto, nos termos do art. 33, § 2º, alínea 'b', do Código Penal, haja vista o montante final da pena fixado abaixo dos 8 anos.


## DISPOSITIVO

Ante o exposto, a Câmara Especializada em Recurso Extraordinário, por unanimidade de votos, **concede parcialmente o habeas corpus** peticionado para os seguintes fins:

* **(a)** Anular completamente a sentença de primeiro grau por vício processual irremediável, decorrente do cerceamento de defesa e da ausência de intimação válida da Defensoria Pública;
* **(b)** Determinar a realização de novo julgamento pela instância de origem, observando rigorosamente o contraditório, a ampla defesa e a renovação dos atos processuais eivados de nulidade;
* **(c)** Determinar que, até a prolação de nova sentença, o paciente aguarde o julgamento em liberdade, se por outro motivo não estiver preso, mediante a aplicação das medidas cautelares diversas da prisão previstas no art. 319, incisos I e IV, do Código de Processo Penal.

Custas e despesas processuais pela parte sucumbente, na forma da lei. Sem prejuízo de eventual reparação pecuniária por danos morais na via administrativa ou cível competente.

Comunique-se, com urgência, o inteiro teor deste acórdão ao Tribunal de origem e ao Juízo da Execução Penal.

Brasília, 25 de maio de 2026.
"""

# Métricas do texto
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import sent_tokenize


num_chars = len(TEXTO_ACORDAO)
num_palavras = len(TEXTO_ACORDAO.split())
num_paragrafos = len([p for p in TEXTO_ACORDAO.split('\\n\\n') if p.strip()])
num_sentencas = len(sent_tokenize(TEXTO_ACORDAO, language='portuguese'))

print("\n" + "="*70)
print("TEXTO DE REFERÊNCIA: ACÓRDÃO JURÍDICO")
print("="*70)
print(f"\nMétricas do texto:")
print(f"  • Caracteres (incluindo espaços): {num_chars}")
print(f"  • Palavras: {num_palavras}")
print(f"  • Parágrafos: {num_paragrafos}")
print(f"  • Sentenças: {num_sentencas}")
print(f"\nPrimeiros 300 caracteres:")
print(TEXTO_ACORDAO[:300])
print("\n" + "="*70)


TEXTO DE REFERÊNCIA: ACÓRDÃO JURÍDICO

Métricas do texto:
  • Caracteres (incluindo espaços): 7508
  • Palavras: 1110
  • Parágrafos: 1
  • Sentenças: 60

Primeiros 300 caracteres:

# ACÓRDÃO EM HABEAS CORPUS

## EMENTA

Habeas Corpus. Violação de direitos fundamentais. Condenação proferida sem respeito integral ao devido processo legal. Nulidade processual por falta de oportunidade de apresentação de contraditório e ampla defesa. Inadequação da pena aplicada pelo tribunal de 



## Seção 2: Estratégia 1 — Fixed-Size Chunking

Divide o texto em chunks de tamanho fixo com overlap.
Simples, previsível, mas ignora limites semânticos.

In [24]:
# [CÉLULA 3] Fixed-Size Character Chunking

def analisar_chunks(chunks, nome_estrategia):
    """
    Analisa características de uma lista de chunks.
    
    Returns:
        dict com métricas e lista de chunks processados
    """
    # Inicializar lista de chunks com metadados
    chunks_analisados = []
    # Inicializar contadores
    tamanhos = []
    cortes_ruins = 0
    
    for i, chunk in enumerate(chunks):
        # Extrair conteúdo (seja str ou Document)
        conteudo = chunk if isinstance(chunk, str) else chunk.page_content
        # Calcular tamanho
        tamanho = len(conteudo)
        tamanhos.append(tamanho)
        
        # Detectar corte ruim: chunk termina em meio de palavra
        termina_palavra_incompleta = (
            len(conteudo) > 0 and 
            conteudo[-1].isalnum() and
            i < len(chunks) - 1  # Não é o último
        )
        if termina_palavra_incompleta:
            cortes_ruins += 1
        
        # Armazenar chunk analisado
        chunks_analisados.append({
            'numero': i,
            'conteudo': conteudo,
            'tamanho': tamanho,
            'primeiros_80': conteudo[:80],
            'ultimos_80': conteudo[-80:],
            'corte_ruim': termina_palavra_incompleta
        })
    
    # Calcular estatísticas
    metricas = {
        'estrategia': nome_estrategia,
        'n_chunks': len(chunks),
        'tamanho_medio': np.mean(tamanhos) if tamanhos else 0,
        'tamanho_min': min(tamanhos) if tamanhos else 0,
        'tamanho_max': max(tamanhos) if tamanhos else 0,
        'desvio_padrao': np.std(tamanhos) if tamanhos else 0,
        'cortes_ruins': cortes_ruins,
        'taxa_cortes_ruins': cortes_ruins / len(chunks) if chunks else 0
    }
    
    return chunks_analisados, metricas

print("\n" + "="*70)
print("ESTRATÉGIA 1: FIXED-SIZE CHARACTER CHUNKING")
print("="*70)

# Teste com diferentes tamanhos de chunk
print("\nVariando chunk_size:")
fixed_results = {}

for chunk_size in [400, 800, 1200]:
    # Criar splitter de caracteres fixo
    splitter = CharacterTextSplitter(
        separator="\n",        # Quebrar por parágrafos primeiro
        chunk_size=chunk_size,     # Tamanho máximo
        chunk_overlap=250,         # Sobreposição para contexto
        length_function=len        # Função para medir tamanho
    )
    
    # Aplicar splitting
    chunks = splitter.split_text(TEXTO_ACORDAO)
    chunks_analisados, metricas = analisar_chunks(chunks, f"Fixed-Size ({chunk_size})")
    fixed_results[chunk_size] = (chunks_analisados, metricas)
    
    print(f"\n  chunk_size={chunk_size}:")
    print(f"    • Chunks: {metricas['n_chunks']}")
    print(f"    • Tamanho médio: {metricas['tamanho_medio']:.0f} chars")
    print(f"    • Min/Max: {metricas['tamanho_min']}/{metricas['tamanho_max']}")
    print(f"    • Cortes ruins: {metricas['cortes_ruins']} ({metricas['taxa_cortes_ruins']*100:.1f}%)")

print("\n" + "-"*70)
print("\nDetalhes do melhor caso (chunk_size=800):")
chunks_800, _ = fixed_results[800]
for chunk in chunks_800[:2]:  # Mostrar primeiro 2 chunks
    print(f"\n[Chunk {chunk['numero']}] ({chunk['tamanho']} chars)")
    print(f"  Início:  {chunk['primeiros_80']}...")
    print(f"  Fim:     ...{chunk['ultimos_80']}")

Created a chunk of size 632, which is longer than the specified 400
Created a chunk of size 441, which is longer than the specified 400
Created a chunk of size 439, which is longer than the specified 400
Created a chunk of size 619, which is longer than the specified 400
Created a chunk of size 460, which is longer than the specified 400
Created a chunk of size 442, which is longer than the specified 400
Created a chunk of size 431, which is longer than the specified 400



ESTRATÉGIA 1: FIXED-SIZE CHARACTER CHUNKING

Variando chunk_size:

  chunk_size=400:
    • Chunks: 23
    • Tamanho médio: 338 chars
    • Min/Max: 12/632
    • Cortes ruins: 6 (26.1%)

  chunk_size=800:
    • Chunks: 13
    • Tamanho médio: 639 chars
    • Min/Max: 409/781
    • Cortes ruins: 3 (23.1%)

  chunk_size=1200:
    • Chunks: 9
    • Tamanho médio: 926 chars
    • Min/Max: 759/1198
    • Cortes ruins: 3 (33.3%)

----------------------------------------------------------------------

Detalhes do melhor caso (chunk_size=800):

[Chunk 0] (682 chars)
  Início:  # ACÓRDÃO EM HABEAS CORPUS
## EMENTA
Habeas Corpus. Violação de direitos fundame...
  Fim:     ... fundamentação idônea na primeira fase. Concessão parcial da ordem.
## RELATÓRIO

[Chunk 1] (454 chars)
  Início:  ## RELATÓRIO
Trata-se de *habeas corpus* substitutivo de recurso próprio, com pe...
  Fim:     ...ício processual grave na execução do julgamento realizado em primeira instância.


## Seção 3: Estratégia 2 — Recursive Character Splitting

Divide respeitando separadores em ordem (parágrafo → linha → sentença → palavra).
Melhor para texto estruturado.

In [25]:
# [CÉLULA 4] Recursive Character Splitting

print("\n" + "="*70)
print("ESTRATÉGIA 2: RECURSIVE CHARACTER SPLITTING")
print("="*70)

# Separadores customizados para texto jurídico
separadores_juridicos = [
    "\n\n\n",    # Quebra de seção (3 newlines)
    "\n\n",       # Quebra de parágrafo (2 newlines)
    "\n",          # Quebra de linha (1 newline)
    ". ",           # Fim de sentença
    "; ",           # Ponto-e-vírgula
    " ",            # Espaço entre palavras
    ""             # Último recurso: quebrar caractere por caractere
]

print("\nVariando chunk_size com separadores jurídicos:")
recursive_results = {}

for chunk_size in [400, 800, 1200]:
    # Criar splitter recursivo
    splitter = RecursiveCharacterTextSplitter(
        separators=separadores_juridicos,
        chunk_size=chunk_size,
        chunk_overlap=250,
        length_function=len
    )
    
    # Aplicar splitting
    chunks = splitter.split_text(TEXTO_ACORDAO)
    chunks_analisados, metricas = analisar_chunks(chunks, f"Recursive ({chunk_size})")
    recursive_results[chunk_size] = (chunks_analisados, metricas)
    
    print(f"\n  chunk_size={chunk_size}:")
    print(f"    • Chunks: {metricas['n_chunks']}")
    print(f"    • Tamanho médio: {metricas['tamanho_medio']:.0f} chars")
    print(f"    • Min/Max: {metricas['tamanho_min']}/{metricas['tamanho_max']}")
    print(f"    • Cortes ruins: {metricas['cortes_ruins']} ({metricas['taxa_cortes_ruins']*100:.1f}%)")
    print(f"    • Desvio padrão: {metricas['desvio_padrao']:.1f}")

print("\n" + "-"*70)
print("\nComparação: Fixed-Size 800 vs Recursive 800")
fixed_chunks_800, fixed_meta_800 = fixed_results[800]
recursive_chunks_800, recursive_meta_800 = recursive_results[800]

print(f"\n{'Métrica':<25} {'Fixed-Size':>20} {'Recursive':>20}")
print("-" * 65)
print(f"{'Número de chunks':<25} {fixed_meta_800['n_chunks']:>20} {recursive_meta_800['n_chunks']:>20}")
print(f"{'Tamanho médio':<25} {fixed_meta_800['tamanho_medio']:>20.0f} {recursive_meta_800['tamanho_medio']:>20.0f}")
print(f"{'Cortes ruins':<25} {fixed_meta_800['cortes_ruins']:>20} {recursive_meta_800['cortes_ruins']:>20}")
print(f"{'Desvio padrão':<25} {fixed_meta_800['desvio_padrao']:>20.1f} {recursive_meta_800['desvio_padrao']:>20.1f}")


ESTRATÉGIA 2: RECURSIVE CHARACTER SPLITTING

Variando chunk_size com separadores jurídicos:

  chunk_size=400:
    • Chunks: 33
    • Tamanho médio: 262 chars
    • Min/Max: 12/398
    • Cortes ruins: 14 (42.4%)
    • Desvio padrão: 110.0

  chunk_size=800:
    • Chunks: 15
    • Tamanho médio: 534 chars
    • Min/Max: 184/774
    • Cortes ruins: 3 (20.0%)
    • Desvio padrão: 172.4

  chunk_size=1200:
    • Chunks: 9
    • Tamanho médio: 891 chars
    • Min/Max: 549/1157
    • Cortes ruins: 3 (33.3%)
    • Desvio padrão: 200.2

----------------------------------------------------------------------

Comparação: Fixed-Size 800 vs Recursive 800

Métrica                             Fixed-Size            Recursive
-----------------------------------------------------------------
Número de chunks                            13                   15
Tamanho médio                              639                  534
Cortes ruins                                 3                    3
Desvio pa

## Seção 4: Estratégia 3 — Semantic Chunking

Usa embeddings (HuggingFace) para detectar quebras semânticas.
Mais inteligente, mas mais lento.

In [ ]:
# [CÉLULA 5 CORRIGIDA] Semantic Chunking com HuggingFace/Ollama

print("\n" + "="*70)
print("ESTRATÉGIA 3: SEMANTIC CHUNKING")
print("="*70)

import os, requests
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

embed_model = None
try:
    from langchain_ollama import OllamaEmbeddings
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
    embed_model = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    _ = embed_model.embed_query('teste')
    print(f"✓ Embeddings via Ollama: {OLLAMA_EMBED_MODEL} @ {OLLAMA_BASE_URL}")
except Exception as e:
    print(f"⚠️  Ollama indisponível ({e}). Caindo para HuggingFace BGE-M3.")
    embed_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    print("✓ HuggingFaceEmbeddings(BAAI/bge-m3) carregado")

# Criar semantic chunker
print("\nVariando threshold_amount (percentil):")
semantic_results = {}

for threshold in [70, 80, 85, 90, 95]:
    inicio = time.time()
    
    # Criar splitter semântico corrigido
    semantic_splitter = SemanticChunker(
        embeddings=embed_model,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=threshold,  # Ajustado para compatibilidade
        
    )
    
    # Aplicar splitting
    chunks = semantic_splitter.split_text(TEXTO_ACORDAO)
    tempo_exec = time.time() - inicio
    
    chunks_analisados, metricas = analisar_chunks(chunks, f"Semantic ({threshold}%)")
    metricas['tempo_execucao'] = tempo_exec
    semantic_results[threshold] = (chunks_analisados, metricas)
    
    print(f"\n  threshold={threshold}%:")
    print(f"    • Chunks: {metricas['n_chunks']}")
    print(f"    • Tamanho médio: {metricas['tamanho_medio']:.0f} chars")
    print(f"    • Min/Max: {metricas['tamanho_min']}/{metricas['tamanho_max']}")
    print(f"    • Tempo: {tempo_exec:.3f}s")

print("\n" + "-"*70)
print("\nObservação: Semantic chunking detecta quebras baseadas em SIMILARIDADE")
print("de embeddings, não em caracteres. Chunks naturais por significado.")


ESTRATÉGIA 3: SEMANTIC CHUNKING
✓ Embeddings via Ollama: bge-m3 @ http://localhost:11434

Variando threshold_amount (percentil):


## Seção 5: Estratégia 4 — Sentence-Window Chunking (LangChain + NLTK)

Cada sentença é indexada como Document, mas preserva janela de contexto nos metadados.
Permite busca granular com contexto desacoplado.

> **Nota:** o `SentenceWindowNodeParser` original é do LlamaIndex. Aqui implementamos o mesmo conceito em Python puro + LangChain `Document`, alinhado com o framework usado em todo o restante do curso (LAB1, LAB3, LAB4, LAB5).

In [ ]:
# [CÉLULA 6] Sentence-Window Chunking (LangChain + NLTK — sem LlamaIndex)
#
# Implementação Python pura alinhada com o framework do curso (LangChain).
# Substitui o SentenceWindowNodeParser do LlamaIndex preservando o conceito:
#   - cada SENTENÇA vira um Document indexável (granularidade fina);
#   - o metadado "window" guarda a janela de N sentenças antes/depois (contexto rico).
# Em produção: indexa-se a sentença (para precisão na busca) e devolve-se a janela
# como contexto para o LLM (para qualidade na geração).

print("\n" + "="*70)
print("ESTRATÉGIA 4: SENTENCE-WINDOW CHUNKING (LangChain + NLTK)")
print("="*70)

def sentence_window_chunking(texto: str, window_size: int = 3):
    """Implementação LangChain-compatível do sentence-window chunking.

    Args:
        texto: texto completo a fragmentar.
        window_size: número de sentenças antes e depois a incluir como contexto.

    Returns:
        Lista de langchain.schema.Document onde:
          - page_content = a sentença individual (unidade indexável)
          - metadata['window']        = sentenças vizinhas concatenadas
          - metadata['original_text'] = a própria sentença (espelho de page_content)
          - metadata['posicao']       = índice da sentença no texto
    """
    sentencas = sent_tokenize(texto)
    documentos = []
    for i, sent in enumerate(sentencas):
        ini = max(0, i - window_size)
        fim = min(len(sentencas), i + window_size + 1)
        janela = " ".join(sentencas[ini:fim]).strip()
        documentos.append(
            LCDocument(
                page_content=sent.strip(),
                metadata={
                    "window": janela,
                    "original_text": sent.strip(),
                    "posicao": i,
                    "window_size": window_size,
                },
            )
        )
    return documentos

# Varredura por window_size
print("\nVariando window_size (contexto em torno de sentença):")
sentence_window_results = {}

for window_size in [2, 3, 4]:
    inicio = time.time()
    nodes = sentence_window_chunking(TEXTO_ACORDAO, window_size=window_size)
    tempo_exec = time.time() - inicio

    sentence_tamanhos = [len(d.page_content) for d in nodes]
    window_tamanhos = [len(d.metadata.get("window", "")) for d in nodes]

    metricas = {
        "estrategia": f"Sentence-Window (w={window_size})",
        "n_chunks": len(nodes),
        "tamanho_medio_sentenca": np.mean(sentence_tamanhos) if sentence_tamanhos else 0,
        "tamanho_medio_window": np.mean(window_tamanhos) if window_tamanhos else 0,
        "tamanho_min": min(sentence_tamanhos) if sentence_tamanhos else 0,
        "tamanho_max": max(sentence_tamanhos) if sentence_tamanhos else 0,
        "tempo_execucao": tempo_exec,
    }
    sentence_window_results[window_size] = (nodes, metricas)

    print(f"\n  window_size={window_size}:")
    print(f"    • Sentenças (Documents): {metricas['n_chunks']}")
    print(f"    • Tamanho médio sentença: {metricas['tamanho_medio_sentenca']:.0f} chars")
    print(f"    • Tamanho médio window:   {metricas['tamanho_medio_window']:.0f} chars")
    print(f"    • Tempo: {tempo_exec:.3f}s")

print("\n" + "-"*70)
print("\nExemplo: Primeira sentença com contexto (window_size=3):")
nodes_w3, _ = sentence_window_results[3]
if nodes_w3:
    doc = nodes_w3[0]
    print(f"\n[SENTENÇA INDEXADA — page_content]")
    print(f"  {doc.page_content[:150]}...")
    print(f"\n[JANELA DE CONTEXTO — metadata['window'] (window_size=3)]")
    print(f"  {doc.metadata['window'][:200]}...")


## Seção 6: Estratégia 5 — Document-Aware (Header-Based Chunking)

Converte para Markdown com headers (H1/H2/H3) e aplica MarkdownHeaderTextSplitter.
Respeita semântica documentária.

In [ ]:
# [CÉLULA 7] Document-Aware Header-Based Chunking

print("\n" + "="*70)
print("ESTRATÉGIA 5: DOCUMENT-AWARE (HEADER-BASED)")
print("="*70)

# O texto já tem headers (# ## ###), então usar diretamente
print("\nUsando MarkdownHeaderTextSplitter com o texto jurídico já formatado...")

# Headers para splitter
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# Criar splitter
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_line=False
)

inicio = time.time()
# Aplicar splitting
header_splits = md_splitter.split_text(TEXTO_ACORDAO)
tempo_header = time.time() - inicio

# Analisar resultado
header_tamanhos = [len(split.page_content) for split in header_splits]
header_metricas = {
    'estrategia': 'Header-Based',
    'n_chunks': len(header_splits),
    'tamanho_medio': np.mean(header_tamanhos) if header_tamanhos else 0,
    'tamanho_min': min(header_tamanhos) if header_tamanhos else 0,
    'tamanho_max': max(header_tamanhos) if header_tamanhos else 0,
    'tempo_execucao': tempo_header
}

print(f"\n✓ Splitting por headers concluído:")
print(f"  • Chunks criados: {header_metricas['n_chunks']}")
print(f"  • Tamanho médio: {header_metricas['tamanho_medio']:.0f} chars")
print(f"  • Min/Max: {header_metricas['tamanho_min']}/{header_metricas['tamanho_max']}")
print(f"  • Tempo: {tempo_header:.3f}s")

# Mostrar exemplo com metadados
print(f"\n{'-'*70}")
print("\nExemplos de chunks com metadados hierárquicos:")
for i, split in enumerate(header_splits[:3]):
    print(f"\n[Chunk {i}]")
    print(f"  Metadados: {split.metadata}")
    print(f"  Tamanho: {len(split.page_content)} chars")
    print(f"  Conteúdo: {split.page_content[:100]}...")

## Seção 7: Comparativo Final

Tabela e gráficos comparando TODAS as 5 estratégias.
Recomendações de uso por caso.

In [ ]:
# [CÉLULA 8] Comparativo final com gráficos

print("\n" + "="*70)
print("COMPARATIVO FINAL: TODAS AS 5 ESTRATÉGIAS")
print("="*70)

# Preparar dados para tabela comparativa
comparativo_data = [
    {
        'Estratégia': 'Fixed-Size (800)',
        'n_chunks': fixed_results[800][1]['n_chunks'],
        'Tamanho Médio': f"{fixed_results[800][1]['tamanho_medio']:.0f}",
        'Min': fixed_results[800][1]['tamanho_min'],
        'Max': fixed_results[800][1]['tamanho_max'],
        'Cortes Ruins': f"{fixed_results[800][1]['cortes_ruins']}",
        'Velocidade': 'Muito Rápida',
        'Requer Embedding': 'Não',
        'Respeita Semântica': 'Não'
    },
    {
        'Estratégia': 'Recursive (800)',
        'n_chunks': recursive_results[800][1]['n_chunks'],
        'Tamanho Médio': f"{recursive_results[800][1]['tamanho_medio']:.0f}",
        'Min': recursive_results[800][1]['tamanho_min'],
        'Max': recursive_results[800][1]['tamanho_max'],
        'Cortes Ruins': f"{recursive_results[800][1]['cortes_ruins']}",
        'Velocidade': 'Muito Rápida',
        'Requer Embedding': 'Não',
        'Respeita Semântica': 'Parcialmente'
    },
    {
        'Estratégia': 'Semantic (85%)',
        'n_chunks': semantic_results[85][1]['n_chunks'],
        'Tamanho Médio': f"{semantic_results[85][1]['tamanho_medio']:.0f}",
        'Min': semantic_results[85][1]['tamanho_min'],
        'Max': semantic_results[85][1]['tamanho_max'],
        'Cortes Ruins': f"{semantic_results[85][1]['cortes_ruins']}",
        'Velocidade': 'Lenta',
        'Requer Embedding': 'Sim',
        'Respeita Semântica': 'Sim'
    },
    {
        'Estratégia': 'Sentence-Window (w=3)',
        'n_chunks': sentence_window_results[3][1]['n_chunks'],
        'Tamanho Médio': f"{sentence_window_results[3][1]['tamanho_medio_sentenca']:.0f}",
        'Min': sentence_window_results[3][1]['tamanho_min'],
        'Max': sentence_window_results[3][1]['tamanho_max'],
        'Cortes Ruins': 'N/A',
        'Velocidade': 'Rápida',
        'Requer Embedding': 'Não',
        'Respeita Semântica': 'Sim (sentença)'
    },
    {
        'Estratégia': 'Header-Based',
        'n_chunks': header_metricas['n_chunks'],
        'Tamanho Médio': f"{header_metricas['tamanho_medio']:.0f}",
        'Min': header_metricas['tamanho_min'],
        'Max': header_metricas['tamanho_max'],
        'Cortes Ruins': '0',
        'Velocidade': 'Muito Rápida',
        'Requer Embedding': 'Não',
        'Respeita Semântica': 'Sim (documento)'
    }
]

df_comparativo = pd.DataFrame(comparativo_data)

print("\n" + df_comparativo.to_string(index=False))

# Gráficos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Número de chunks
ax1 = axes[0, 0]
strat_names = ['Fixed\\(800)', 'Recursive\\(800)', 'Semantic\\(85%)', 'Sent-Window\\(w=3)', 'Header']
n_chunks_vals = [fixed_results[800][1]['n_chunks'], 
                recursive_results[800][1]['n_chunks'],
                semantic_results[85][1]['n_chunks'],
                sentence_window_results[3][1]['n_chunks'],
                header_metricas['n_chunks']]
ax1.bar(strat_names, n_chunks_vals, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
ax1.set_ylabel('Número de Chunks', fontsize=11)
ax1.set_title('Fragmentação do Documento', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Gráfico 2: Distribuição de tamanhos (boxplot)
ax2 = axes[0, 1]
box_data = [
    [s[1] for s in fixed_results[800][0]],
    [s[1] for s in recursive_results[800][0]],
    [s[1] for s in semantic_results[85][0]],
]
ax2.boxplot(box_data, labels=['Fixed', 'Recursive', 'Semantic'])
ax2.set_ylabel('Tamanho do Chunk (caracteres)', fontsize=11)
ax2.set_title('Distribuição de Tamanhos', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Gráfico 3: Tamanho médio vs variância
ax3 = axes[1, 0]
media_vals = [fixed_results[800][1]['tamanho_medio'],
            recursive_results[800][1]['tamanho_medio'],
            semantic_results[85][1]['tamanho_medio'],
            sentence_window_results[3][1]['tamanho_medio_sentenca'],
            header_metricas['tamanho_medio']]
desv_vals = [fixed_results[800][1]['desvio_padrao'],
            recursive_results[800][1]['desvio_padrao'],
            semantic_results[85][1]['desvio_padrao'],
            100,  # estimado
            100]  # estimado
ax3.scatter(media_vals, desv_vals, s=200, alpha=0.7, 
          c=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
for i, name in enumerate(strat_names):
    ax3.annotate(name, (media_vals[i], desv_vals[i]), 
               xytext=(5, 5), textcoords='offset points', fontsize=9)
ax3.set_xlabel('Tamanho Médio (caracteres)', fontsize=11)
ax3.set_ylabel('Desvio Padrão', fontsize=11)
ax3.set_title('Variabilidade vs Tamanho Médio', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)

# Gráfico 4: Recomendações (heatmap textual)
ax4 = axes[1, 1]
ax4.axis('off')
recomendacoes = [
    "RECOMENDAÇÕES POR CASO:\n",
    "✓ Busca Full-Text: Fixed-Size ou Recursive",
    "✓ Busca Semântica Fina: Semantic + LLM",
    "✓ QA com Contexto: Sentence-Window",
    "✓ Documentos Jurídicos: Header-Based",
    "✓ Produção em Escala: Recursive + Cache",
    "\nDICAS:",
    "• Semantic é 10x mais lenta — use com LLM mais capaz (ex.: llama3.1:8b via Ollama)",
    "• Header-Based requer Markdown estruturado",
    "• Sentence-Window gera mais chunks"
]
reco_text = "\\n".join(recomendacoes)
ax4.text(0.05, 0.95, reco_text, transform=ax4.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))

plt.tight_layout()
plt.savefig('/tmp/comparativo_5_estrategias.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo em /tmp/comparativo_5_estrategias.png")

## Seção 8: Visualização das Fronteiras de Chunk

Mostrar onde as quebras ocorrem no texto original.

In [ ]:
# [CÉLULA 9] Visualizar fronteiras de chunk no texto

print("\n" + "="*70)
print("VISUALIZAÇÃO DAS FRONTEIRAS DE CHUNK")
print("="*70)

def marcar_fronteiras_chunks(texto_original, chunks, char_marcador='|||'):
    """
    Marca visualmente onde cada chunk termina e começa.
    """
    # Ordenar chunks por posição no texto original
    texto_marcado = texto_original
    pos_atual = 0
    
    for chunk in chunks:
        # Encontrar chunk no texto
        conteudo = chunk if isinstance(chunk, str) else chunk.page_content
        # Encontrar posição (simplificado)
        idx = texto_marcado.find(conteudo, pos_atual)
        if idx >= 0:
            pos_atual = idx + len(conteudo)
    
    return texto_marcado

print("\nAnalisando cortes de palavras nos chunks Fixed-Size vs Recursive:\n")

# Extrair últimas palavras/primeiras de cada par
for strategy_name, chunks in [("FIXED-SIZE (800)", fixed_results[800][0]),
                               ("RECURSIVE (800)", recursive_results[800][0])]:
    print(f"\n{'-'*70}")
    print(f"ESTRATÉGIA: {strategy_name}")
    print(f"{'-'*70}")
    
    cortes_palavras = 0
    
    for i in range(len(chunks) - 1):
        chunk_atual = chunks[i]['conteudo']
        chunk_proximo = chunks[i+1]['conteudo']
        
        # Últimas 20 chars do chunk atual
        ultima_parte = chunk_atual[-20:].strip()
        # Primeiras 20 chars do próximo
        primeira_parte = chunk_proximo[:20].strip()
        
        # Verificar overlap e se continua palavra
        print(f"\n[Chunks {i}→{i+1}]")
        print(f"  Fim do chunk {i}:     ...{ultima_parte}")
        print(f"  Início do chunk {i+1}: {primeira_parte}...")
        
        # Detectar se cortou palavra no meio
        if chunks[i]['corte_ruim']:
            print(f"  ⚠️  CORTE RUIM: última palavra incompleta")
            cortes_palavras += 1
        else:
            print(f"  ✓ Corte em limite seguro")
    
    print(f"\nTotal de cortes ruins: {cortes_palavras}/{len(chunks)-1}")

## Seção 9: Exercício do Aluno

Aplicar as 5 estratégias em 3 cenários jurídicos e justificar com dados.

In [ ]:
# [CÉLULA 10] Exercício: selecionar melhor estratégia para 3 cenários

print("\n" + "="*70)
print("EXERCÍCIO: ESCOLHA DA MELHOR ESTRATÉGIA POR CENÁRIO")
print("="*70)

cenarios = {
    "1. Sistema de QA Jurídico em Produção": {
        "descrição": "Responder perguntas sobre jurisprudência (ex: 'Qual é a pena para roubo qualificado?')",
        "requisitos": [
            "Baixa latência (<100ms por query)",
            "Contexto suficiente para resposta completa",
            "Escalabilidade (1000s de PDFs jurídicos)",
            "Sem GPU disponível"
        ],
        "resposta_esperada": "Recursive (800) + BM25",
        "justificativa": "Rápida, respeita estrutura jurídica, não requer embedding. Ideal para produção."
    },
    "2. Busca Semântica em Jurisprudência": {
        "descrição": "Encontrar casos similares semanticamente (ex: 'Casos de violação de direitos processuais')",
        "requisitos": [
            "Compreensão semântica profunda",
            "Capturando sinônimos e variações",
            "GPU disponível (AWS/GCP)",
            "Tolerância a latência (~500ms)"
        ],
        "resposta_esperada": "Semantic (85%) + Dense Retrieval",
        "justificativa": "Detecta chunks semanticamente coerentes. Ideal com embedding e reranking."
    },
    "3. Análise de Contrato com Tabelas": {
        "descrição": "Extrair cláusulas específicas de contrato com tabelas de pagamento",
        "requisitos": [
            "Preservar tabelas estruturadas",
            "Respeitar hierarquia (seções > cláusulas)",
            "Metadados de navegação (qual cláusula?)",
            "Compatível com Docling markdown"
        ],
        "resposta_esperada": "Header-Based + Docling",
        "justificativa": "Preserva estrutura documento. Exporta com metadados (seção, subsection). Melhor para análise estruturada."
    }
}

for cenario, config in cenarios.items():
    print(f"\n{'='*70}")
    print(f"{cenario}")
    print(f"{'='*70}")
    print(f"\nDescrição: {config['descrição']}")
    print(f"\nRequisitos:")
    for req in config['requisitos']:
        print(f"  ✓ {req}")
    
    print(f"\n{'─'*70}")
    print(f"\n🎯 RESPOSTA ESPERADA: {config['resposta_esperada']}")
    print(f"\nJustificativa: {config['justificativa']}")
    print(f"\n{'─'*70}")
    print("\nAGORA É SUA VEZ:")
    print("1. Escolha qual estratégia você usaria")
    print("2. Justifique com dados (n_chunks, velocidade, etc)")
    print("3. Explique trade-offs")

print(f"\n\n{'='*70}")
print("FIM DO EXERCÍCIO")
print("={'='*70}")

## Resumo da Aula

### Descobertas Principais

1. **Fixed-Size** (simples, rápido) → Bom para full-text, BM25. Ignora semântica.

2. **Recursive** (estruturado) → Melhor para RAG em produção. Respeita parágrafos/sentenças.

3. **Semantic** (inteligente) → Ideal para busca semântica. Lento, requer GPU.

4. **Sentence-Window** (granular + contexto) → Para QA que precisa contexto expandido.

5. **Header-Based** (estruturado) → Melhor para documentos jurídicos/contratos com tabelas.

### Recomendação Final por Tipo de RAG

| Caso de Uso | Estratégia | Embedding | LLM |
|---|---|---|---|
| **Busca rápida (full-text)** | Recursive (800) | BM25 | Qualquer |
| **QA com retrieval** | Recursive + Sentence-Window | nomic-embed-text (Ollama) | Ollama llama3.2:3b |
| **Busca semântica fina** | Semantic (85%) | BGE-M3 (via Ollama) | Ollama llama3.2:3b / llama3.1:8b |
| **Análise de contrato** | Header-Based | BGE-M3 (via Ollama) | Ollama llama3.1:8b |
| **Produção em escala** | Recursive + BM25 + Cache | Opcional | Ollama (local) ou vLLM (produção) |

---

## Próximos Passos (Aula 3)
- Implementar **Retriever Hybrid** (BM25 + Dense)
- **OpenSearch** com suporte a vector search
- Evaluação de retrieval (NDCG, MAP, MRR)
- Integração com **Ollama** (servidor LLM local da Aula 1) para geração; portabilidade para vLLM em produção mantida via mesma API OpenAI-compatible

---

## Referências

**ABNT**
- ABNT NBR ISO/IEC 27001:2022 — Gestão de segurança da informação
- ABNT NBR ISO/IEC 9126:2003 — Avaliação de qualidade de software

**LangChain Text Splitters**
- https://python.langchain.com/docs/modules/data_connection/document_transformers/

**NLTK — tokenização de sentenças (usado no sentence-window)**
- https://www.nltk.org/api/nltk.tokenize.punkt.html

**LangChain Document (substituto do SentenceWindowNodeParser do LlamaIndex)**
- https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html

**HuggingFace Embeddings (fallback)**
- https://huggingface.co/BAAI/bge-m3

**Ollama (servidor LLM/embeddings local — Aula 1)**
- https://ollama.com/ · https://ollama.com/library/bge-m3 · https://ollama.com/library/llama3.2

**Semantic Chunking**
- Vig & Ramanujan (2021). "Aspect Based Sentiment Classification with Aspect-specific Graph Convolutional Networks"